# 02 — Ingredient Extractor\n\nThis notebook demonstrates the `IngredientExtractor` module, which combines\nspaCy NER with vocabulary-based n-gram matching to identify food ingredients\nin free text.

## 1. Setup and Vocabulary Stats

In [1]:
import json
import os
import sys
from pathlib import Path

# Ensure the backend root is on sys.path so 'pantry' is importable
_backend = str(Path.cwd().parent) if Path.cwd().name == "notebooks" else str(Path.cwd())
if _backend not in sys.path:
    sys.path.insert(0, _backend)

from pantry.ingredient_extractor import IngredientExtractor

# Load the extractor (pass absolute vocab path for notebook context)
_vocab_path = str(Path(_backend) / "data" / "processed" / "ingredient_vocab.json")
extractor = IngredientExtractor(vocab_path=_vocab_path)

# Vocabulary stats
with open(_vocab_path) as f:
    vocab_raw = json.load(f)

print(f"Vocabulary size:      {len(vocab_raw):,} ingredients")
print(f"Single-word entries:  {sum(1 for k in vocab_raw if ' ' not in k):,}")
print(f"Multi-word entries:   {sum(1 for k in vocab_raw if ' ' in k):,}")
print(f"\nTop 15 ingredients by frequency:")
for i, (name, count) in enumerate(list(vocab_raw.items())[:15], 1):
    print(f"  {i:2d}. {name:<35s} ({count:,})")

Vocabulary size:      4,675 ingredients
Single-word entries:  311
Multi-word entries:   4,364

Top 15 ingredients by frequency:
   1. kosher salt                         (2,633)
   2. sugar                               (2,536)
   3. salt                                (2,499)
   4. olive oil                           (1,901)
   5. fresh lemon juice                   (1,536)
   6. large egg                           (1,472)
   7. freshly ground black pepper         (1,442)
   8. extra-virgin olive oil              (1,413)
   9. water                               (1,273)
  10. all-purpose flour                   (1,234)
  11. unsalted butter                     (1,155)
  12. vegetable oil                       (996)
  13. baking powder                       (832)
  14. vanilla extract                     (827)
  15. whole milk                          (725)


## 2. Example Prompts\n\nRun the extractor on a diverse set of inputs — simple lists, full sentences,\nand varied cuisines.

In [2]:
prompts = [
    # Simple lists
    "chicken, garlic, salt, and olive oil",
    "butter, flour, sugar, milk, and egg",

    # Full sentences
    "I want to make a stir fry with rice, soy sauce, ginger, and sesame oil",
    "Can you suggest a recipe using tomatoes, basil, and garlic?",
    "I have leftover chicken, onion, and celery in the fridge",

    # Multi-word ingredients
    "I need coconut milk, bay leaf, and black pepper for the curry",
    "Drizzle some olive oil and add a pinch of paprika",

    # Plural forms
    "I bought tomatoes, onions, and carrots at the market",
    "Slice the potatoes and sprinkle with thyme and oregano",

    # Baking context
    "Mix the flour, sugar, butter, vanilla extract, and baking powder",

    # Minimal / sparse
    "Just salt and pepper please",
    "I only have rice",

    # No ingredients
    "Tell me a fun fact about cooking",
    "What temperature should I set the oven to?",

    # Mixed with non-ingredient words
    "My grandmother's recipe calls for honey, cinnamon, and nutmeg with love",
]

for prompt in prompts:
    result = extractor.extract(prompt)
    print(f"INPUT:  {prompt}")
    print(f"FOUND:  {result}")
    print()

INPUT:  chicken, garlic, salt, and olive oil
FOUND:  ['chicken', 'garlic', 'garlic salt', 'oil', 'olive', 'olive oil', 'salt']

INPUT:  butter, flour, sugar, milk, and egg
FOUND:  ['butter', 'egg', 'flour', 'milk', 'sugar']

INPUT:  I want to make a stir fry with rice, soy sauce, ginger, and sesame oil
FOUND:  ['ginger', 'oil', 'rice', 'sesame oil', 'soy sauce']

INPUT:  Can you suggest a recipe using tomatoes, basil, and garlic?
FOUND:  ['basil', 'garlic', 'tomato']

INPUT:  I have leftover chicken, onion, and celery in the fridge
FOUND:  ['celery', 'chicken', 'onion']

INPUT:  I need coconut milk, bay leaf, and black pepper for the curry
FOUND:  ['bay leaf', 'black pepper', 'coconut milk', 'milk', 'pepper']

INPUT:  Drizzle some olive oil and add a pinch of paprika
FOUND:  ['oil', 'olive', 'olive oil', 'paprika']

INPUT:  I bought tomatoes, onions, and carrots at the market
FOUND:  ['carrot', 'onion', 'tomato']

INPUT:  Slice the potatoes and sprinkle with thyme and oregano
FOUND:  [

## 3. Edge Cases

In [3]:
edge_cases = {
    "Empty string":          "",
    "Whitespace only":       "   ",
    "All caps":              "CHICKEN AND GARLIC",
    "Mixed case":            "Olive Oil and Butter",
    "Repeated ingredient":   "garlic garlic garlic",
    "Ingredient substring":  "salted caramel",  # should "salt" match?
}

for label, text in edge_cases.items():
    result = extractor.extract(text)
    print(f"{label:25s} | input={text!r:40s} | found={result}")

Empty string              | input=''                                       | found=[]
Whitespace only           | input='   '                                    | found=[]
All caps                  | input='CHICKEN AND GARLIC'                     | found=['chicken', 'garlic']
Mixed case                | input='Olive Oil and Butter'                   | found=['butter', 'oil', 'olive', 'olive oil']
Repeated ingredient       | input='garlic garlic garlic'                   | found=['garlic']
Ingredient substring      | input='salted caramel'                         | found=[]


## 4. Vocabulary Distribution

In [4]:
import collections

counts = list(vocab_raw.values())
word_counts = [len(k.split()) for k in vocab_raw]
wc = collections.Counter(word_counts)

print(f"Frequency range:  {min(counts)} – {max(counts)}")
print(f"Median frequency: {sorted(counts)[len(counts)//2]}")
print()
print("Ingredient length distribution (number of words):")
for n_words in sorted(wc):
    print(f"  {n_words} word(s): {wc[n_words]:,} entries ({100*wc[n_words]/len(vocab_raw):.1f}%)")

Frequency range:  3 – 2633
Median frequency: 5

Ingredient length distribution (number of words):
  0 word(s): 1 entries (0.0%)
  1 word(s): 310 entries (6.6%)
  2 word(s): 1,197 entries (25.6%)
  3 word(s): 1,327 entries (28.4%)
  4 word(s): 922 entries (19.7%)
  5 word(s): 445 entries (9.5%)
  6 word(s): 255 entries (5.5%)
  7 word(s): 120 entries (2.6%)
  8 word(s): 60 entries (1.3%)
  9 word(s): 18 entries (0.4%)
  10 word(s): 10 entries (0.2%)
  11 word(s): 2 entries (0.0%)
  13 word(s): 2 entries (0.0%)
  14 word(s): 2 entries (0.0%)
  15 word(s): 1 entries (0.0%)
  16 word(s): 2 entries (0.0%)
  18 word(s): 1 entries (0.0%)
